# Step 1. Framing the Problem

Some students leave college before finishing their degree. A school that spots who is at
risk early can help them before they drop out. The best time to spot them is around the
time they enroll.

This notebook sets up the problem so the rest of the project starts on solid ground. It
answers four questions.

1. **What is the real problem, and who does it help?**
2. **What kind of machine learning task is this, and why that kind?**
3. **How is success measured, in model terms and in money terms?**
4. **What data does the model use, what is left out, and why?**

Each answer carries into later steps. The choices here shape how the data is prepared in
Step 3. They also shape how the model is checked for fairness in Step 5.

In [1]:
import sys
from pathlib import Path

# I locate the repo root once, so src/ becomes importable from notebooks/.
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "src" / "paths.py").exists():
        sys.path.insert(0, str(_p))
        break

import pandas as pd
from src.paths import RAW_DIR

In [2]:
# RAW_DIR comes from src.paths and resolves no matter where I launch the
# notebook from. The dataset uses ';' as its separator, not ',', so sep=';'
# is required.
df = pd.read_csv(RAW_DIR / "dropout.csv", sep=";")

# I check what loaded. Table size, and how the outcome splits.
print("rows, columns", df.shape)
print()
print(df["Target"].value_counts())

rows, columns (4424, 37)

Target
Graduate    2209
Dropout     1421
Enrolled     794
Name: count, dtype: int64


## 1. The Problem, and Who It Helps

Every year, some students leave college before finishing their degree. The cost falls on
three sides. The student loses time, money, and future earnings. The school loses
tuition and the money it spent on that student. Society loses part of the skilled
workforce it helped train.

The data comes from the Polytechnic Institute of Portalegre (IPP), a public college in
Portugal. IPP already uses this data in a tool. The tool helps its tutoring team find
students at risk of leaving early, so help can reach them in time.

This project copies that use. The model looks only at what is known when a student
enrolls. From that, it estimates how likely the student is to drop out. It helps the
tutoring staff. They cannot watch every student closely, so they need to spend their
limited time on the students most at risk. The model takes a long list of students and
turns it into a short ranked one. At the top are the students worth contacting first.

The model answers one question. Which students should the support team contact early? A
correct flag reaches a student who would otherwise slip away. A wrong flag wastes staff
time on a student who was fine. It can also miss a student who needed help.

## 2. Scope and Population

The data covers students at one school, IPP. It spans 17 undergraduate degrees and runs
from the 2008 to 2009 year through 2018 to 2019. Every student sits inside the
Portuguese system. Each one is enrolled in a public college and follows its rules on
tuition, scholarships, and admissions.

This sets a clear boundary. The model fits a Portuguese polytechnic like IPP. It is not
a general dropout predictor for any school in any country. A model trained on one school
learns that school's patterns. It learns its mix of students, its local economy, and its
particular years. Those patterns do not carry cleanly to a different school somewhere
else.

The method carries even though the model does not. In this project I frame the problem,
audit the data for bias, compare models, and check fairness. Those steps work for any
school with similar data. This honesty shapes every claim I make. It returns in Step 5
as a stated limit on how far the results reach.

## 3. What the Model Sees, and What It Leaves Out

The dataset holds two kinds of information. The first kind is known when a student
enrolls. It includes the student's age, admission grade, earlier schooling, scholarship
status, and family background. The second kind is recorded later, during the first and
second semesters. It includes how many courses the student took, passed, and failed, and
the grades they earned.

The second kind stays out on purpose. How a student does in their semesters predicts
dropout almost perfectly. That is because a student who is failing courses is often
already on the way out. A model built on this would look very accurate and help no one.
It would only name a dropout once the dropout was already under way. The tool exists to
warn before that point, so anything recorded after enrollment is left out.

This is a deliberate trade. The model uses only what is known at enrollment. So its raw
accuracy is lower than a model that reads semester results. I accept the lower accuracy.
An early warning that is roughly right beats a late confirmation that is almost certain.
Only the early warning leaves time to act.

There is a name for the mistake I am avoiding here. It is called leakage, which means
information about the final outcome sneaks into the inputs the model learns from. The
semester results are a form of leakage, since they already reflect who is leaving. The
same choice returns in Step 3, where the semester columns are dropped in the code. It
returns again in Step 5, where it becomes an accepted limit on accuracy.

## 4. The Task, and Why This Kind

The model sorts each student into one of two groups. Either likely to drop out or likely
to graduate. Sorting things into two groups like this is called binary classification.

The original data has three outcomes. These are Dropout, Graduate, and Enrolled.
Enrolled means the student was still studying when the data was recorded, so we do not
yet know how their story ends. A student with no final outcome cannot teach the model
what dropout looks like. So the Enrolled students are removed. That leaves two settled
outcomes, which fits a simple yes or no question.

Other kinds of task do not fit. Regression is out because it predicts a number, and the
answer we want is a category, not a quantity. Clustering is out because it groups data
with no labels, but the labels are known here, dropout or graduate. Recommendation is
out because it suggests items to a user, which is a different problem. Our outcome is a
settled category with a clear label. So classification is the right choice.

In [3]:
# The task is binary, Dropout vs Graduate. Enrolled has no settled outcome, so I drop it.
binary = df[df["Target"].isin(["Dropout", "Graduate"])].copy()
binary["dropout"] = (binary["Target"] == "Dropout").astype(int)

n = len(binary)
rate = binary["dropout"].mean()

print("rows after dropping Enrolled", n)
print("dropout rate", round(rate * 100, 1), "percent")
print()
print(binary["Target"].value_counts())

rows after dropping Enrolled 3630
dropout rate 39.1 percent

Target
Graduate    2209
Dropout     1421
Name: count, dtype: int64


## 5. How Success Is Measured

Two kinds of measure matter here. How well the model works, and what it is worth.

The model does not give a plain yes or no. Instead it gives a probability. It might say
a student is 0.82 likely to drop out, rather than just dropout. To turn that number into
a decision, we need a line that splits flag from do not flag. That dividing line is
called the cutoff, and **the cutoff is a choice, not a given**. Move it and every count
changes. So the measures we use to judge the model should be ones that hold no matter
where the cutoff sits. The cutoff itself gets picked on its own, on purpose, in Step 4.

Accuracy is the obvious measure, and it misleads here. About 39 in 100 students in the
two outcome groups drop out, as the numbers above show. A lazy model that calls everyone
a graduate would be right about 61 percent of the time. Yet it would catch no one at
risk. High accuracy, no value. So I report accuracy but I do not treat it as the
headline number.

Two measures pick the model, because neither one depends on where the cutoff sits. Both
are a kind of score called AUC, which is short for area under the curve. Think of it as
a single number that sums up how the model does across every possible cutoff at once.

The first is PR AUC. It is built from two ideas, precision and recall. Precision asks,
of the students I flag, how many really do drop out. Recall asks, of the students who
really drop out, how many did I flag. PR AUC combines these two across every cutoff. It
ignores the graduates the model correctly leaves alone. That keeps it honest when the
two groups are uneven in size, which they are here. This is the measure I tune the model
on.

The second is ROC AUC. Picture taking one student who dropped out and one who graduated,
both at random. ROC AUC is the chance that the model gives the dropout the higher risk
score. It reads like a ranking score. A 1.0 means perfect order and a 0.5 means a coin
flip. It matches the real use, where staff work down a ranked list.

One more measure judges the finished system, but only once the cutoff exists. That
measure is recall on the dropout group. Of the students who really drop out, how many
did the model flag? A missed student is the costly mistake, since that student loses the
help the tool was built to provide. This is what the school actually cares about. It is
the number I would report to them.

But recall on its own cannot pick a model, and it is worth being clear about why.
**Recall is not a property of a model. It is a property of a model and a cutoff
together.** Give a recall figure without saying where the cutoff sits and you have said
almost nothing. I can push recall to 1.0 on any model at all, just by flagging every
single student. So PR AUC and ROC AUC choose the model. Step 4 then chooses the cutoff,
using the value numbers in the next section. Only then does recall mean something.

## 6. The Business Value

The model earns its place by saving more than it costs. The value has a simple shape. It
is the worth of the students we keep, minus the cost of reaching out to the students we
flag.

I set three assumptions. I state them plainly so the estimate stays honest.

First, a kept student is worth about 2,100 euros. This comes from the Portuguese public
tuition cap, which is roughly 697 euros a year over about three years. I set it low on
purpose. Tuition is a direct figure I can point to. The wider loss to the student and to
society is larger, but it is harder to defend with a firm number.

Second, reaching out does not guarantee keeping the student. A flag starts a
conversation. It does not keep the student on its own. So I treat 2,100 euros as a
ceiling for each student we catch. To be more realistic, I scale it down by how often
outreach actually works. For example, we might keep 3 in 10 of the at risk students we
reach.

Third, the cost of reaching out is only an estimate. I report it across a range, roughly
50, 150, or 300 euros per flagged student. The real figure depends on the school.

This keeps the value honest. It rewards catching at risk students. It charges for
flagging too many. And it never claims a result the model alone cannot deliver.

These three numbers are not decoration. Step 4 uses them directly to choose the
probability cutoff. That is the one place in this project where cost and benefit
actually meet. A cutoff picked without them is a cutoff picked by accident.

## What Step 1 Settles

Four decisions carry into the rest of the project.

1. The task is binary classification, Dropout vs Graduate, with the Enrolled students
   removed.
2. The model uses only what is known at enrollment. It drops all semester records to
   avoid leakage.
3. The model is chosen on PR AUC and ROC AUC, since these hold no matter where the
   cutoff sits. Recall is the number the school cares about, but it only means something
   once a cutoff is fixed. So Step 4 fixes one on purpose. Accuracy is reported but not
   trusted.
4. Value is the students we keep minus the cost of outreach. That is about 2,100 euros a
   student against 50 to 300 euros a contact, with the assumptions stated above. Step 4
   turns these numbers into the cutoff.

Step 3 acts on the leakage decision in code. Step 4 picks the cutoff using the value
numbers above. Step 5 revisits the scope limit and the fairness of the flags.